# Land cash flow and IRR check

This page re-runs Landscape's land development cash flow on a real project, then checks the result against the raw records it was built from. Every number below was recalculated when this page was produced. Nothing is written to the database.

**What it checks**

1. **Costs.** Does every budget line and the land purchase reach the cash flow?
2. **Revenue.** Does every parcel sale reach the cash flow, and do gross price, deductions and net proceeds add up?
3. **Which rows feed the IRR.** The exact rows the engine adds into the IRR, and a check that they total the reported profit.
4. **IRR by method.** The same cash flow run through each IRR calculation in the code, to see whether they agree.

Each check ends in **PASS** or **FLAG**. A flag means the numbers don't tie out and someone has to decide what's correct. It doesn't necessarily mean the engine is wrong.

In [ ]:
# --- setup (hidden in the rendered page) ---
import os, sys, datetime as dt
from collections import defaultdict
import pandas as pd, numpy as np, numpy_financial as npf
from IPython.display import display, Markdown, HTML

REPO = os.environ.get('LANDSCAPE_REPO', os.path.expanduser('~/mnt/landscape'))
sys.path.insert(0, os.path.join(REPO, 'backend'))
os.chdir(os.path.join(REPO, 'backend'))
from dotenv import load_dotenv
load_dotenv(os.path.join(REPO, '.cowork-django-env'))
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"   # notebooks run inside an event loop
import django; django.setup()
from django.db import connection
try:
    from loguru import logger as _lg; _lg.remove()   # keep engine debug logging off the page
except Exception: pass
from apps.financial.services.land_dev_cashflow_service import LandDevCashFlowService

PROJECT_ID = int(os.environ.get('CHECK_PROJECT_ID', 9))
TOL = 1.0   # dollars: anything within $1 counts as tying out

def q(sql, args=None):
    with connection.cursor() as c:
        c.execute(sql, args or [])
        cols = [d[0] for d in c.description]
        return pd.DataFrame(c.fetchall(), columns=cols)

def money(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return '—'
    x = float(x)
    return f'({abs(x):,.0f})' if x < -0.5 else f'{abs(x) if abs(x) < 0.5 else x:,.0f}'

def pct(x): return '—' if x is None else f'{x*100:.2f}%'

FINDINGS = []
def verdict(name, ok, detail):
    FINDINGS.append((name, 'PASS' if ok else 'FLAG', detail))
    colour = '#1d6b3a' if ok else '#9a3b0c'
    display(HTML(f'<p style="margin:6px 0"><b style="color:{colour}">{"PASS" if ok else "FLAG"}</b> — {detail}</p>'))

def show(df, money_cols=(), pct_cols=()):
    fmt = {c: money for c in money_cols}; fmt.update({c: pct for c in pct_cols})
    num = set(money_cols) | set(pct_cols)
    txt = [c for c in df.columns if c not in num and not pd.api.types.is_numeric_dtype(df[c])]
    st = df.style.format(fmt).hide(axis='index').set_properties(subset=txt, **{'text-align': 'left'})
    st = st.set_table_styles([{'selector': f'th.col{i}', 'props': 'text-align:left'} for i, c in enumerate(df.columns) if c in txt], overwrite=False)
    display(st)

proj = q('select project_name from landscape.tbl_project where project_id=%s', [PROJECT_ID]).iloc[0, 0]
svc = LandDevCashFlowService(PROJECT_ID)
R = svc.calculate(include_financing=False)
RF = LandDevCashFlowService(PROJECT_ID).calculate(include_financing=True)
A = svc._get_dcf_assumptions()
S = R['summary']
NOW = dt.datetime.now(dt.timezone(dt.timedelta(hours=-7)))
display(Markdown(f"**Project:** {proj} (#{PROJECT_ID}) · **Run:** {NOW:%Y-%m-%d %I:%M %p} Phoenix · "
                 f"**Live database, read-only** · **Horizon:** {R['startDate']} to {R['endDate']} ({R['totalPeriods']} months)"))

## What the engine reports

In [ ]:
rows = [('Gross revenue', S['totalGrossRevenue']), ('Revenue deductions', -S['totalRevenueDeductions']),
        ('Net revenue', S['totalNetRevenue']), ('Total costs', -S['totalCosts']), ('Gross profit', S['grossProfit']),
        ('Peak equity', S['peakEquity'])]
show(pd.DataFrame(rows, columns=['Item', 'Amount']), money_cols=['Amount'])
display(Markdown(f"IRR **{pct(S['irr'])}** · NPV at {pct(A.get('discount_rate'))} **{money(S['npv'])}** · "
                 f"equity multiple **{S['equityMultiple']:.2f}x** · price growth {pct(A.get('price_growth_rate'))} · "
                 f"cost inflation {pct(A.get('cost_inflation_rate'))}"))

## 1. Costs

The engine reads the project budget and the land purchase, spreads each line over its months and applies cost inflation. Two things are checked here. First, does every budget line make it in? (The engine skips lines of zero or below.) Second, does any line fall outside the cash flow's time span, where it would be counted in the cost total but left out of the cash flow?

In [ ]:
bud = q('''select b.fact_id, coalesce(c.category_name,'Uncategorized') as category, b.activity, b.amount,
                 b.start_period, b.periods_to_complete
          from landscape.core_fin_fact_budget b
          left join landscape.core_unit_cost_category c on c.category_id=b.category_id
          where b.project_id=%s order by b.fact_id''', [PROJECT_ID])
bud['amount'] = bud['amount'].astype(float)
skipped = bud[~(bud['amount'] > 0)]
acq_raw = q('select coalesce(sum(amount),0) as amt from landscape.tbl_acquisition where project_id=%s', [PROJECT_ID]).iloc[0,0]

cost_sections = [s for s in R['sections'] if s['sectionId'].startswith('cost-')]
eng_lines = [(s['sectionName'], li) for s in cost_sections for li in s['lineItems']]
eng_total = sum(li['total'] for _, li in eng_lines)
in_window = sum(pv['amount'] for _, li in eng_lines for pv in li['periods'] if 0 <= pv['periodIndex'] < R['totalPeriods'])

summary = pd.DataFrame([
    ('Budget lines in the database', str(len(bud)), bud['amount'].sum()),
    ('   of which skipped by the engine (zero or below)', str(len(skipped)), skipped['amount'].sum()),
    ('Land purchase in the database', '', float(acq_raw)),
    ('Engine cost total, after inflation', '', S['totalCosts']),
    ('Engine costs that land inside the cash flow months', '', abs(in_window)),
], columns=['Item', 'Lines', 'Amount'])
show(summary, money_cols=['Amount'])
infl = S['totalCosts'] - bud.loc[bud['amount']>0,'amount'].sum() - float(acq_raw)
display(Markdown(f"Cost inflation adds **{money(infl)}** on top of the raw budget and land purchase."))
verdict('Every budget line reaches the cash flow', len(skipped) == 0,
        f"{len(skipped)} budget line(s) skipped" if len(skipped) else 'all budget lines are positive and included')
verdict('No costs fall outside the cash flow months', abs(eng_total - in_window) <= TOL,
        f"costs inside the months {money(abs(in_window))} vs cost total {money(abs(eng_total))}; difference {money(abs(eng_total - in_window))}")

## 2. Revenue

Each parcel's sale comes from its stored sale assumptions: gross price, improvement offset, commission, closing costs and net proceeds. The engine grows the price by the price-growth rate from the project start to the month of sale. Checked here: which parcels never reach the cash flow, and whether gross revenue less deductions equals net revenue.

In [ ]:
par = q('''select p.parcel_code, p.type_code, p.units_total, p.acres_gross, p.sale_period,
                 psa.gross_parcel_price as gross, psa.improvement_offset_total as offset,
                 psa.commission_amount as comm,
                 coalesce(psa.legal_amount,0)+coalesce(psa.closing_cost_amount,0)+coalesce(psa.title_insurance_amount,0) as closing,
                 psa.net_sale_proceeds as net
          from landscape.tbl_parcel p left join landscape.tbl_parcel_sale_assumptions psa using(parcel_id)
          where p.project_id=%s order by p.parcel_code''', [PROJECT_ID])
for c in ['gross','offset','comm','closing','net','units_total','acres_gross']:
    par[c] = par[c].astype(float)

def why_dropped(r):
    if not ((r.units_total or 0) > 0 or (r.acres_gross or 0) > 0): return 'no units or acres'
    if pd.isna(r.sale_period): return 'no sale month'
    if pd.isna(r.gross) or pd.isna(r.net): return 'no sale assumptions'
    return ''
par['dropped'] = par.apply(why_dropped, axis=1)
dropped = par[par['dropped'] != '']
display(Markdown(f"**{len(par)}** parcels in the project; **{len(par)-len(dropped)}** are sold in the cash flow."))
if len(dropped):
    dropped = dropped.assign(acres_gross=dropped['acres_gross'].map(lambda v: f'{v:,.1f}'))
    show(dropped[['parcel_code','type_code','acres_gross','dropped']].rename(columns={
        'parcel_code':'Parcel','type_code':'Use','acres_gross':'Acres','dropped':'Not in the cash flow because'}))
verdict('Every parcel with land is either sold or explained', all(dropped['type_code'].isin(['PARK','OS','ROW','SCH'])) ,
        'parcels left out: ' + (', '.join(f"{r.parcel_code} ({r.type_code}, {r.dropped})" for r in dropped.itertuples()) or 'none'))

sold = par[par['dropped'] == ''].copy()
sold['implied_net'] = sold['gross'] - sold['offset'].fillna(0) - sold['comm'].fillna(0) - sold['closing']
sold['gap'] = sold['net'] - sold['implied_net']
bad = sold[sold['gap'].abs() > TOL]
verdict('Stored net proceeds = gross − offset − commission − closing, parcel by parcel', len(bad) == 0,
        f"{len(bad)} of {len(sold)} parcels don't tie; largest gap {money(bad['gap'].abs().max()) if len(bad) else 0}")

g = A.get('price_growth_rate') or 0
sold['esc'] = (1 + g) ** ((sold['sale_period'] - 1) / 12)
gap_total = S['totalGrossRevenue'] - S['totalRevenueDeductions'] - S['totalNetRevenue']
offset_flat = sold['offset'].fillna(0).sum()
offset_escalated = (sold['offset'].fillna(0) * sold['esc']).sum()
growth_part = offset_escalated - offset_flat
missing_part = offset_flat - S['totalSubdivisionCosts']   # offsets on parcels not priced per front foot
tbl = pd.DataFrame([
    ('Gross revenue (grown)', S['totalGrossRevenue']),
    ('Less deductions the engine reports', -S['totalRevenueDeductions']),
    ('= what net revenue should be', S['totalGrossRevenue'] - S['totalRevenueDeductions']),
    ('Net revenue the engine uses in the cash flow', S['totalNetRevenue']),
    ('Difference', -gap_total),
    ('   explained by: growth applied to offsets in net revenue but not in deductions', -growth_part),
    ('   explained by: offsets on parcels not priced per front foot, left out of deductions', -missing_part),
    ('   unexplained', -(gap_total - growth_part - missing_part)),
], columns=['Item', 'Amount'])
show(tbl, money_cols=['Amount'])
display(Markdown(f"**What this means.** The cash flow, and so the IRR, subtracts improvement offsets grown at {pct(g)} a year, on every parcel that has one. "
                 f"The revenue summary shows them flat, and only on parcels priced per front foot. The IRR isn't affected. What's wrong is the revenue "
                 f"summary a user reads: it overstates what net revenue should be by **{money(gap_total)}**."))
verdict('Gross revenue − deductions = net revenue', abs(gap_total) <= TOL,
        f"off by {money(gap_total)}: {money(growth_part)} from growth on offsets, {money(missing_part)} from offsets left out of deductions, "
        f"{money(gap_total - growth_part - missing_part)} unexplained")

## 3. Which rows feed the IRR

The engine builds the IRR from one monthly cash flow line. It adds up every row in the cash flow except gross parcel sales and the separate deduction rows, because net revenue already includes both. Below is every row the engine adds or leaves out, and a check that the rows it adds total the reported gross profit.

In [ ]:
def feeds(li): return li.get('subcategory') not in ('Parcel Sales', 'Revenue Deductions')
rows = []
for s in RF['sections']:
    for li in s['lineItems']:
        rows.append((s['sectionName'].title(), li.get('description') or li.get('category'), li.get('containerLabel') or '',
                     li['total'], 'yes' if feeds(li) else 'no'))
feed = pd.DataFrame(rows, columns=['Section', 'Row', 'Where', 'Total', 'In the IRR'])
show(feed, money_cols=['Total'])
mon = [0.0]*R['totalPeriods']
for s in R['sections']:
    for li in s['lineItems']:
        if feeds(li):
            for pv in li['periods']:
                if 0 <= pv['periodIndex'] < len(mon): mon[pv['periodIndex']] += pv['amount']
verdict('Rows feeding the IRR total the reported gross profit', abs(sum(mon) - S['grossProfit']) <= TOL,
        f"IRR rows {money(sum(mon))} vs gross profit {money(S['grossProfit'])}")
fin = [li for s in RF['sections'] if s['sectionId']=='financing' for li in s['lineItems']]
fin_total = sum(li['total'] for li in fin)
verdict('Loan rows stay out of the project IRR', abs(fin_total) <= TOL or RF['summary']['irr'] == S['irr'],
        f"loan rows total {money(fin_total)}; IRR with loans shown {pct(RF['summary']['irr'])} vs without {pct(S['irr'])}")

## 4. IRR by method

The same monthly cash flow is run through each IRR calculation the code has, so any disagreement is between the methods, not the inputs. Landscape reports the IRR on actual dates: each month's cash counts when it arrives (decided 2026-10-01). For reference, the page also shows the old yearly method. That method added each calendar year's months into one figure and counted it as arriving at the start of the year, which pulls late-year sales forward and raises the IRR. The yearly row is left out of the agreement check.

In [ ]:
import importlib
periods = R['periods']
dates = [dt.date.fromisoformat(p['startDate'][:10]) if isinstance(p['startDate'], str) else p['startDate'] for p in periods]
years = defaultdict(float)
for d, cf in zip(dates, mon): years[d.year] += cf
annual = [years[y] for y in sorted(years)]
monthly_irr = npf.irr(mon); monthly_ann = (1+monthly_irr)**12 - 1 if not np.isnan(monthly_irr) else None
out = [('Engine as shipped', S['irr'])]

out.append(('Monthly, then annualized', monthly_ann))
from financial_engine.waterfall.irr import calculate_xirr
from decimal import Decimal
x = calculate_xirr(dates, [Decimal(str(round(c, 2))) for c in mon])
out.append(('Exact dates (the waterfall\'s XIRR)', float(x) if x is not None else None))
from apps.calculations.series_metrics import irr_from_series
sm = irr_from_series(mon); out.append(('Signed-series helper on months, annualized', (1+sm)**12-1 if sm is not None else None))
irrs = pd.DataFrame(out, columns=['Method', 'IRR'])
vals = [v for v in irrs['IRR'] if v is not None]
irrs = pd.concat([irrs, pd.DataFrame([('Reference only: old yearly method', float(npf.irr(annual)))], columns=['Method', 'IRR'])])
show(irrs, pct_cols=['IRR'])
spread = max(vals) - min(vals)
verdict('IRR methods agree within half a point', spread <= 0.005, f"spread between the engine and the other methods {spread*100:.2f} points")

# Where the money actually arrives vs where the yearly method puts it
tim = []
for i, y in enumerate(sorted(years)):
    idx = [k for k, d in enumerate(dates) if d.year == y]
    pos = [(k, mon[k]) for k in idx if mon[k] > 0]
    w = sum(c for _, c in pos)
    avg_month = (sum(k*c for k, c in pos) / w) if w else None
    tim.append((y, years[y], i*12, None if avg_month is None else round(avg_month, 1)))
tdf = pd.DataFrame(tim, columns=['Year', 'Net cash flow', 'Month the old yearly method used', 'Month the inflows actually arrive (cash-weighted)'])
tdf['Month the inflows actually arrive (cash-weighted)'] = tdf['Month the inflows actually arrive (cash-weighted)'].map(lambda v: '—' if v is None or pd.isna(v) else f'{v:.1f}')
show(tdf, money_cols=['Net cash flow'])
display(Markdown("Months are counted from the first month of the cash flow (month 0). Which IRR to put in front of investors is a business decision; this page only shows how far apart the two are."))
dr = A.get('discount_rate') or 0
m_rate_at_irr = (1 + S['irr'])**(1/12) - 1
npv_at_irr = npf.npv(m_rate_at_irr, mon)
verdict('IRR and NPV use the same timing (NPV at the reported IRR is about zero)', abs(npv_at_irr) < 0.001*S['totalCashOut'],
        f"discounting the monthly cash flow at {pct(S['irr'])}, the reported IRR, leaves {money(npv_at_irr)} instead of zero" + ("" if abs(npv_at_irr) < 0.001*S['totalCashOut'] else ". NPV and IRR are timed differently"))

## Results

In [ ]:
res = pd.DataFrame(FINDINGS, columns=['Check', 'Result', 'Detail'])
display(res.style.hide(axis='index').set_properties(**{'text-align':'left'}).map(lambda v: 'color:#9a3b0c;font-weight:600' if v=='FLAG' else ('color:#1d6b3a;font-weight:600' if v=='PASS' else ''), subset=['Result']))
display(Markdown(f"**{(res.Result=='PASS').sum()} passed, {(res.Result=='FLAG').sum()} flagged.** Project #{PROJECT_ID}, run {NOW:%Y-%m-%d %I:%M %p} Phoenix."))

---
*How to re-run:* run the land check script in this folder; it re-executes this notebook against the live database and writes the page again. To check a different project, set CHECK_PROJECT_ID.